
# Experiment Summary Aggregator (Notebook)
Dieses Notebook liest eine **Experiment_Summary**-CSV aus einem Basisordner ein  
und **hängt zusätzliche Daten** (Pfade & Metadaten) über *run_id* / *quant_mode* an, 
indem es die zugehörigen `ErrorMetrics_all_runs.csv` sowie JSON-Metrikdateien findet und verarbeitet.

## Was es macht
- Liest die Datei `Experiment_Summary_Serevr_multiconfig.csv` (oder kompatible Varianten) aus einem Basisordner.
- Vereinheitlicht Spalten (z. B. `level` vs. `profile`).
- Sucht pro `run_id` die Datei `Error_Metrics/ErrorMetrics_all_runs.csv` und joinet die passenden `json_path` / `predictions_file_path` je nach `quant_mode` an.
- Liest die JSON-Fehlerdateien und hängt **zusätzliche Metadaten** (Dataset, Zeitstempel, Trainings- & Inferenz-Config, Modellfilename etc.) an.
- Optional: berechnet aus den Metrik-Arrays sinnvolle Kennzahlen (z. B. Mittelwerte je Metrik und je gewünschtem `horizon`-Schritt).
- Speichert ein **angereichertes CSV** unter `Analysis/Experiment_Aggregated_Summary_enriched.csv` (wird automatisch angelegt).

> Hinweis: Dieses Notebook ist robust gegenüber beiden Summary-Varianten (Version 1/2).  
> Bitte die erste Zelle mit den **Parametern** anpassen und die Zellen nacheinander ausführen.


In [16]:

# === PARAMETER ===
# Basisordner der Output-Struktur (ohne abschließenden Backslash):
base_path = r"C:\DEV\RevPi_ML\ML_Edge_Device\Output"
base_path = r"C:\DEV\RevPi_ML\zwischenergebnisse_3"

# Dateiname der zu lesenden Summary:
summary_filename = "Experiment_Summary_Serevr_multiconfig.csv"
summary_filename = "Experiment_Summary_Serevr_multiconfig_RevPi.csv"

# Ausgabeordner (wird bei Bedarf erstellt)
analysis_subdir = "Analysis"
enriched_csv_name = "Experiment_Aggregated_Summary_enriched.csv"

# === OPTIONEN ===
# Falls True: pro JSON-Metrik-Array auch den Wert zum jeweiligen 'horizon' extrahieren (Index horizon-1).
extract_horizon_specific = True

# Falls True: Mittelwerte je Metrik (über alle Horizonte in der JSON-Liste) mit anhängen.
compute_metric_means = True


In [17]:

import os
import json
import glob
import math
from pathlib import Path
import pandas as pd
from collections import defaultdict

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

def _normalize_quant_mode_from_variant_name(variant_name: str) -> str:
    # Bestimme quant_mode aus einer Variantenbezeichnung wie 'keras', 'model.keras',
    # 'tflite_model_quant_float16', 'model_quant_float16.tflite', 'model_quant_int8.tflite' usw.
    if not isinstance(variant_name, str):
        return ""
    v = variant_name.lower()
    if "int8" in v:
        return "quant-8"
    if "float16" in v or "fp16" in v:
        return "quant-16"
    if "keras" in v:
        return "no-quant"
    # Fallback: kein Treffer -> lieber leer lassen
    return ""

ALGO_TO_SUBDIR = {
    "cnn1d": "CNN1D",
    "lstm": "LSTM",
    "random_forest": "Random_Forest",
    "xgboost": "XGBoost",
    "light_xgboost": "Light_XGBoost",
    "lightgbm": "LightGBM",
}

def _safe_path(*parts) -> Path:
    return Path(*parts)

def _find_all_runs_csv(base: Path, run_id: str, algo: str = None) -> Path | None:
    """
    Suche die Datei 'Error_Metrics/ErrorMetrics_all_runs.csv' zu einem run_id.
    Versucht zuerst den Algorithmus-Ordner, danach globale Suche (langsamer).
    """
    # 1) gezielter Versuch über Algo-Unterordner
    if algo:
        subdir = ALGO_TO_SUBDIR.get(str(algo).lower(), None)
        if subdir:
            candidate = base / subdir / run_id / "Error_Metrics" / "ErrorMetrics_all_runs.csv"
            if candidate.exists():
                return candidate

    # 2) Fallback: globale Suche ab base
    pattern = str(base / "**" / run_id / "Error_Metrics" / "ErrorMetrics_all_runs.csv")
    matches = glob.glob(pattern, recursive=True)
    if matches:
        return Path(matches[0])

    return None

def _read_summary_csv(path: Path) -> pd.DataFrame:
    # sep=None + engine='python' versucht Delimiter zu erkennen (Komma/Semikolon etc.)
    df = pd.read_csv(path, sep=None, engine="python")
    # Vereinheitliche Spaltennamen
    cols = [c.strip() for c in df.columns]
    df.columns = cols
    # Version 2 hat 'profile' statt 'level'
    if "level" not in df.columns and "profile" in df.columns:
        df["level"] = df["profile"]
    # Lowercase algorithm
    if "algorithm" in df.columns:
        df["algorithm"] = df["algorithm"].astype(str).str.lower()
    # quant_mode in einheitliches Format
    if "quant_mode" in df.columns:
        df["quant_mode"] = df["quant_mode"].astype(str).str.lower()
    # Korrigiere Datentypen, wo sinnvoll
    for num_col in ["lags", "horizon"]:
        if num_col in df.columns:
            df[num_col] = pd.to_numeric(df[num_col], errors="coerce")
    return df

def _load_all_runs_df(all_runs_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(all_runs_csv)
    # Erwartete Spalten: run_id, model_name, dataset, time_stamp, model_variant, json_path, predictions_file_path
    # Erzeuge eine Spalte quant_mode_normalized aus model_variant
    if "model_variant" in df.columns:
        df["quant_mode_normalized"] = df["model_variant"].map(_normalize_quant_mode_from_variant_name)
    else:
        df["quant_mode_normalized"] = ""
    return df

def _read_json_safely(path: Path) -> dict:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        return {"__error__": str(e), "__path__": str(path)}

def _flatten_training_inference_config(payload: dict) -> dict:
    # Ziehe aus training_config / inference_config nützliche Felder in flacher Struktur heraus.
    out = {}
    if not isinstance(payload, dict):
        return out

    # run info
    run = payload.get("run", {})
    for k in ["run_id", "model_name", "dataset", "time_stamp"]:
        if k in run:
            out[f"run.{k}"] = run[k]

    # training_config
    tr = payload.get("training_config", {})
    simple_keys = [
        "dataset","train_fraction","validation_fraction","rolling_window_size","lags",
        "batch_size","epochs","learning_rate","optimizer","loss","clipnorm",
        "cnn_dropout","cnn_activation","edge_device","enable_edge","train_time_s",
        "model_size_MB","training_time_s"
    ]
    for k in simple_keys:
        if k in tr:
            out[f"training_config.{k}"] = tr[k]

    # inference_config (nur ein paar Kernfelder)
    inf = payload.get("inference_config", {})
    for k in ["inference_steps","retraining_interval_steps","retraining_cycles",
              "inference_interval_sec","horizon","model_name","model_filename"]:
        if k in inf:
            out[f"inference_config.{k}"] = inf[k]

    # extra info
    extra = payload.get("extra_info", {})
    if isinstance(extra, dict):
        for k,v in extra.items():
            out[f"extra_info.{k}"] = v

    return out

def _extract_metrics(payload: dict, horizon: int | float | None, compute_means=True, extract_specific=True) -> dict:
    # Aus payload['metrics'] Metriken aggregieren.
    # - compute_means: Mittelwert je Metrik-Liste berechnen
    # - extract_specific: Wert an Index (horizon-1), falls vorhanden, ebenfalls anfügen
    out = {}
    metrics = payload.get("metrics", {})
    if not isinstance(metrics, dict):
        return out

    # Liste möglicher Feldnamen (je nach JSON variabel, daher dynamisch)
    for mname, mval in metrics.items():
        if isinstance(mval, list) and mval and all(isinstance(x, (int, float)) or (isinstance(x, float) and math.isnan(x)) for x in mval):
            series = pd.Series(mval, dtype="float64")

            if compute_means:
                out[f"metrics_mean.{mname}"] = float(series.mean(skipna=True))

            if extract_specific and horizon is not None and not pd.isna(horizon):
                idx = int(horizon) - 1
                if 0 <= idx < len(series):
                    out[f"metrics_h{int(horizon)}.{mname}"] = float(series.iloc[idx])
        else:
            # Einzelwerte (z. B. weighted_mae)
            if isinstance(mval, (int, float)) and not isinstance(mval, bool):
                out[f"metrics_value.{mname}"] = float(mval) if not pd.isna(mval) else mval

    return out


In [18]:

# === LADE SUMMARY ===
from pathlib import Path

base = Path(base_path)
summary_path = base / summary_filename
if not summary_path.exists():
    raise FileNotFoundError(f"Summary-Datei nicht gefunden: {summary_path}")

df_sum = _read_summary_csv(summary_path).copy()

# Einheitliche Mindestspalten prüfen
required_cols = ["algorithm","lags","horizon","model_variant","quant_mode","run_id"]
missing = [c for c in required_cols if c not in df_sum.columns]
if missing:
    raise ValueError(f"Fehlende Spalten in Summary: {missing}")

# === JOIN: json_path / predictions_file_path via ErrorMetrics_all_runs.csv ===
cache_all_runs = {}  # run_id -> DataFrame
json_paths = []
pred_paths = []

for idx, row in df_sum.iterrows():
    algo = str(row["algorithm"]).lower() if "algorithm" in row else None
    run_id = str(row["run_id"])
    qmode = str(row["quant_mode"]).lower()

    # Lade/cashe ErrorMetrics_all_runs.csv für diesen run_id
    if run_id not in cache_all_runs:
        all_runs_csv = _find_all_runs_csv(base, run_id, algo=algo)
        if all_runs_csv is None:
            cache_all_runs[run_id] = pd.DataFrame()
        else:
            cache_all_runs[run_id] = _load_all_runs_df(all_runs_csv)

    df_ar = cache_all_runs[run_id]
    json_path_val, pred_path_val = None, None
    if not df_ar.empty:
        # Match per quant_mode
        cand = df_ar[df_ar["quant_mode_normalized"] == qmode]
        if cand.empty:
            # Fallback: versuche exakter Variantenname (robustheit)
            mv = str(row["model_variant"])
            cand = df_ar[df_ar["model_variant"].astype(str).str.lower() == mv.lower()]
        if not cand.empty:
            # Nimm die erste passende Zeile
            json_path_val = cand.iloc[0].get("json_path", None)
            pred_path_val = cand.iloc[0].get("predictions_file_path", None)

    json_paths.append(json_path_val)
    pred_paths.append(pred_path_val)

df_sum["json_path"] = json_paths
df_sum["predictions_file_path"] = pred_paths

print("Anzahl gefundener JSON-Pfade:", df_sum["json_path"].notna().sum())
print("Anzahl gefundener Prediction-Pfade:", df_sum["predictions_file_path"].notna().sum())

# === Lade JSONs & hänge Metadaten + Metrik-Aggregate an ===
flat_rows = []
for idx, row in df_sum.iterrows():
    payload = {}
    jpath = row.get("json_path", None)
    if isinstance(jpath, str) and jpath:
        p = Path(jpath)
        payload = _read_json_safely(p)

    flat = _flatten_training_inference_config(payload)
    metrics_extra = _extract_metrics(payload, horizon=row.get("horizon", None),
                                     compute_means=compute_metric_means,
                                     extract_specific=extract_horizon_specific)
    # Füge die Basiszeile hinzu + Anhänge
    base_data = row.to_dict()
    base_data.update(flat)
    base_data.update(metrics_extra)
    flat_rows.append(base_data)

df_enriched = pd.DataFrame(flat_rows)

# === Ausgabe / Speichern ===
analysis_dir = base / analysis_subdir
analysis_dir.mkdir(parents=True, exist_ok=True)
out_path = analysis_dir / enriched_csv_name
df_enriched.to_csv(out_path, index=False)
print(f"Gespeichert: {out_path}")

# Zeige einen schnellen Überblick
df_enriched.head(10)


Anzahl gefundener JSON-Pfade: 144
Anzahl gefundener Prediction-Pfade: 144
Gespeichert: C:\DEV\RevPi_ML\zwischenergebnisse_3\Analysis\Experiment_Aggregated_Summary_enriched.csv


,algorithm,level,lags,horizon,model_variant,quant_mode,avg_inference_time_ms,avg_total_time_ms,avg_cpu_percent,avg_ram_percent,model_size_mb,run_id,json_path,predictions_file_path
0,cnn1d,simple,20,1,model.keras,no-quant,183.099364,272.931780,25.162414,57.385246,0.3485,2025-08-29_111301_9032_train,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...
1,cnn1d,simple,20,1,model_quant_float16.tflite,quant-16,0.551254,85.597657,25.094267,45.188525,0.3485,2025-08-29_111301_9032_train,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...
2,cnn1d,simple,20,1,model_quant_int8.tflite,quant-8,0.559658,85.443419,25.329956,44.965574,0.3485,2025-08-29_111301_9032_train,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...
3,cnn1d,medium,20,1,model.keras,no-quant,181.345298,269.480218,25.378904,55.085246,0.3485,2025-08-29_112056_4326_train,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...
4,cnn1d,medium,20,1,model_quant_float16.tflite,quant-16,0.531278,85.566270,25.288637,51.219672,0.3485,2025-08-29_112056_4326_train,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...
5,cnn1d,medium,20,1,model_quant_int8.tflite,quant-8,0.466388,85.814346,25.257157,49.681967,0.3485,2025-08-29_112056_4326_train,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...
6,cnn1d,high,20,1,model.keras,no-quant,179.227808,266.226210,25.419390,62.931148,0.3485,2025-08-29_112902_6304_train,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...
7,cnn1d,high,20,1,model_quant_float16.tflite,quant-16,0.468569,85.460014,25.125497,49.644262,0.3485,2025-08-29_112902_6304_train,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...
8,cnn1d,high,20,1,model_quant_int8.tflite,quant-8,0.517908,86.098324,25.309572,49.334426,0.3485,2025-08-29_112902_6304_train,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...
9,cnn1d,simple,20,3,model.keras,no-quant,179.241850,266.940346,25.454417,55.755738,0.3489,2025-08-29_113707_2068_train,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...,/home/pi/ML_Edge_Device/Output/CNN1D/2025-08-2...


In [19]:

# Falls Sie das Ergebnis als Tabelle im Notebook sehen möchten:
try:
    from caas_jupyter_tools import display_dataframe_to_user
    display_dataframe_to_user("Experiment_Aggregated_Summary_enriched", df_enriched)
except Exception as e:
    print("Hinweis:", e)


Hinweis: No module named 'caas_jupyter_tools'


In [26]:
# === SINGLE-KACHEL-METRIKEN (robust) =========================================
import os, glob, json, math, re
from pathlib import Path
import numpy as np
import pandas as pd

# --- Eingaben anpassen ---
BASE = r"C:\DEV\RevPi_ML\ML_Edge_Device\Output"
SUMMARY_FILE = "Experiment_Summary_Serevr_multiconfig.csv"  # dein Summary
OUT_FILE = "Experiment_Enriched_SingleMetricKachel.csv"

summary_path = Path(BASE) / SUMMARY_FILE
df = pd.read_csv(summary_path)

# Horizon robust ermitteln
H_series = pd.to_numeric(df.get("horizon_num", df.get("horizon")), errors="coerce")

# Formatter
def _fmt(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return ""
        return f"{float(x):.6g}"
    except Exception:
        return str(x)

# Modellordner je Algorithmus
FOLDER_BY_ALGO = {"cnn1d": "CNN1D", "lstm": "LSTM", "random_forest": "Random_Forest"}

# --- robuste Variantenerkennung / Tokenisierung ------------------------------
def variant_tokens(model_variant: str) -> list[str]:
    mv = (model_variant or "").lower()
    toks = set()
    if not mv:
        return []
    # Grundtoken
    toks |= {mv, mv.replace(".", "_"), os.path.splitext(mv)[0], os.path.basename(mv)}
    # Spezielle Fälle
    if mv.endswith(".keras") or mv == "model.keras":
        toks |= {"keras", "model.keras", "model_keras"}
    if "int8" in mv:
        toks |= {"int8", "quant_int8", "model_quant_int8", "model_quant_int8.tflite", "tflite", "tflite_int8"}
    if "float16" in mv or "fp16" in mv:
        toks |= {"float16", "fp16", "quant_float16", "model_quant_float16", "model_quant_float16.tflite", "tflite", "tflite_float16"}
    if mv.endswith(".joblib") or "joblib" in mv or "sklearn" in mv:
        toks |= {"joblib", "sklearn"}
    return sorted(toks)

def safe_json_load(p: str):
    try:
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

# JSON-Pfad je Zeile bestimmen (robust)
def resolve_json_path(row) -> str | None:
    # 1) Falls Spalte vorhanden und Datei existiert → direkt verwenden
    if "json_path" in row.index:
        jp = row.get("json_path")
        if isinstance(jp, str) and jp and os.path.isfile(jp):
            return jp

    algo = str(row.get("algorithm","")).lower()
    folder = FOLDER_BY_ALGO.get(algo, algo.upper())
    run_id = str(row.get("run_id") or "")
    if not run_id:
        return None

    # Basisordner
    error_dir = Path(BASE) / folder / run_id / "Error_Metrics"
    if not error_dir.exists():
        # Fallback: suche beliebig tief unter BASE nach Run-Ordner/Error_Metrics
        pattern = str(Path(BASE) / "**" / run_id / "Error_Metrics")
        candidates = [Path(p) for p in glob.glob(pattern, recursive=True) if os.path.isdir(p)]
        if candidates:
            error_dir = candidates[0]
        else:
            return None

    # Token vorbereiten
    level = str(row.get("level","")).lower()
    ds = str(row.get("dataset","mqtt_data_filtered.csv"))
    ds_name = os.path.splitext(os.path.basename(ds))[0].lower()
    mv = str(row.get("model_variant",""))
    mv_tokens = variant_tokens(mv)

    # Erwarteter Name (best case)
    def expected_path():
        # versuche mehrere Suffixvarianten
        suffixes = mv_tokens or ["keras","model.keras","model_quant_int8.tflite","model_quant_float16.tflite","joblib"]
        for suf in suffixes:
            name = f"ErrorMetrics_{run_id}_{algo}_{level}_{ds_name}__{suf.replace('.','_')}.json"
            p = error_dir / name
            if p.is_file():
                return str(p)
        return None

    exp = expected_path()
    if exp:
        return exp

    # 2) Fallback: bestes Match im Ordner suchen
    cand = glob.glob(str(error_dir / "ErrorMetrics_*.json"))
    if not cand:
        return None

    # Scoring: Run-ID sehr hoch gewichten, dann algo/level/ds, dann variant tokens
    def score_for(path: str) -> tuple[int,int]:
        name = os.path.basename(path).lower()
        s = 0
        if run_id.lower() in name: s += 100
        for t in filter(None, [algo, level, ds_name]):
            if t in name: s += 10
        for t in mv_tokens:
            if t in name: s += 3
        return (s, -len(name))  # bei Gleichstand kürzerer Name bevorzugt

    cand.sort(key=lambda p: score_for(p), reverse=True)

    # Validierung: falls möglich, Run-ID im JSON prüfen
    for p in cand[:5]:
        payload = safe_json_load(p)
        rid = (((payload or {}).get("run") or {}).get("run_id") or "").lower()
        if run_id.lower() == rid:
            return p

    return cand[0] if cand else None

METRICS = ["mse","rmse","mae","r2","mape","smape","wape","msle","median_ae","mase","weighted_mae"]

def extract_metrics(json_path: str, H: int) -> dict:
    out = {}
    if not json_path or not os.path.isfile(json_path):
        return out
    payload = safe_json_load(json_path)
    if not payload:
        return out

    # Struktur: { "metrics": { ... } }  (so liegt deine Datei vor)
    metrics = (payload.get("metrics") or {})
    if not isinstance(metrics, dict):
        return out

    for k in METRICS:
        if k not in metrics:
            continue
        v = metrics[k]
        col = f"metrics_value.{k}"

        # Listen → auf H begrenzen, NaN/Inf filtern, als eine Zelle "(v1,v2,...)"
        if isinstance(v, (list, tuple)):
            vals = list(v)
            if isinstance(H, (int, np.integer)) and H > 0:
                vals = vals[:H]
            clean = [x for x in vals if not (isinstance(x, float) and (math.isnan(x) or math.isinf(x)))]
            out[col] = "(" + ",".join(_fmt(x) for x in clean) + ")" if clean else ""

        else:
            # Skalar → NaN/Inf zu leerem Feld
            if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
                out[col] = ""
            else:
                out[col] = _fmt(v)
    return out

rows = []
missing_json = 0
for i, row in df.iterrows():
    jp = resolve_json_path(row)
    if not jp:
        missing_json += 1
    H = H_series.iat[i] if i < len(H_series) else None
    H_int = int(H) if (pd.notna(H) and float(H).is_integer()) else 0
    rows.append(extract_metrics(jp, H_int))

metrics_df = pd.DataFrame(rows)

# Zusammenführen
df_out = pd.concat([df.reset_index(drop=True), metrics_df], axis=1)

# Störende Altspalten entfernen, NEUE metrics_value.* behalten
drop_patterns = [r"^metrics_mean\.", r"^metrics_h\d+\.", r"^metrics_value\..*\.h\d+$"]
to_drop = []
for c in df_out.columns:
    for pat in drop_patterns:
        if re.match(pat, c):
            to_drop.append(c); break
if to_drop:
    df_out = df_out.drop(columns=sorted(set(to_drop)))

# Speichern
out_path = Path(BASE) / "Analysis" / OUT_FILE
out_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_path, index=False, encoding="utf-8")

added_cols = [c for c in df_out.columns if c.startswith("metrics_value.")]
print(f"OK: {len(added_cols)} metrics_value-Spalten hinzugefügt → {out_path}")
if missing_json:
    print(f"Warnung: {missing_json} Zeilen ohne (ableitbaren) JSON-Pfad – Metriken dort leer.")
df_out.head(3)


OK: 11 metrics_value-Spalten hinzugefügt → C:\DEV\RevPi_ML\ML_Edge_Device\Output\Analysis\Experiment_Enriched_SingleMetricKachel.csv


,algorithm,level,lags,horizon,model_variant,quant_mode,avg_inference_time_ms,avg_total_time_ms,avg_cpu_percent,avg_ram_percent,model_size_mb,run_id,metrics_value.mse,metrics_value.rmse,metrics_value.mae,metrics_value.r2,metrics_value.mape,metrics_value.smape,metrics_value.wape,metrics_value.msle,metrics_value.median_ae,metrics_value.mase,metrics_value.weighted_mae
0,cnn1d,simple,20,1,model.keras,no-quant,68.298237,97.968399,8.677668,33.963124,0.3466,2025-08-28_214236_6033_train,1.75417,1.32445,0.914354,0.734469,5.33265e+08,48.2262,36.6938,0.127,0.595845,,0.914354
1,cnn1d,simple,20,1,model_quant_float16.tflite,quant-16,0.121673,26.852211,8.213614,34.000000,0.3466,2025-08-28_214236_6033_train,1.75415,1.32444,0.914391,0.734471,5.33417e+08,48.2267,36.6953,0.127015,0.595889,,0.914391
2,cnn1d,simple,20,1,model_quant_int8.tflite,quant-8,0.221640,28.541122,8.084517,34.000000,0.3466,2025-08-28_214236_6033_train,1.7519,1.32359,0.913227,0.734813,5.33672e+08,48.1982,36.6486,0.126875,0.587666,,0.913227
